In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/hlbt-baku-ml-3-housing-prices/sample_submission.csv
/kaggle/input/competitions/hlbt-baku-ml-3-housing-prices/data_description.txt
/kaggle/input/competitions/hlbt-baku-ml-3-housing-prices/train.csv
/kaggle/input/competitions/hlbt-baku-ml-3-housing-prices/test.csv


In [2]:
df_train = pd.read_csv('/kaggle/input/competitions/hlbt-baku-ml-3-housing-prices/train.csv')
df_test = pd.read_csv('/kaggle/input/competitions/hlbt-baku-ml-3-housing-prices/test.csv')
print(f" train head is:", df_train.head())
print(f" test head is:", df_test.head())

 train head is:     Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0  985          90       RL         75.0    10125   Pave   NaN      Reg   
1  778          20       RL        100.0    13350   Pave   NaN      IR1   
2  708         120       RL         48.0     6240   Pave   NaN      Reg   
3  599          20       RL         80.0    12984   Pave   NaN      Reg   
4  875          50       RM         52.0     5720   Pave   NaN      Reg   

  LandContour Utilities  ... PoolArea PoolQC  Fence MiscFeature MiscVal  \
0         Lvl    AllPub  ...        0    NaN    NaN         NaN       0   
1         Lvl    AllPub  ...        0    NaN  MnPrv         NaN       0   
2         Lvl    AllPub  ...        0    NaN    NaN         NaN       0   
3         Bnk    AllPub  ...        0    NaN    NaN         NaN       0   
4         Lvl    AllPub  ...        0    NaN    NaN         NaN       0   

  MoSold YrSold  SaleType  SaleCondition  SalePrice  
0      8   2009       COD   

In [3]:
y = np.log1p(df_train["SalePrice"])
X = df_train.drop(["SalePrice"], axis=1)
test = df_test.copy()

all_data = pd.concat([X, test], axis=0)

In [4]:
all_data["TotalSF"] = all_data["TotalBsmtSF"] + all_data["1stFlrSF"] + all_data["2ndFlrSF"]
all_data["HouseAge"] = all_data["YrSold"] - all_data["YearBuilt"]
all_data["RemodAge"] = all_data["YrSold"] - all_data["YearRemodAdd"]
all_data["TotalBath"] = (
    all_data["FullBath"]
    + 0.5 * all_data["HalfBath"]
    + all_data["BsmtFullBath"]
    + 0.5 * all_data["BsmtHalfBath"]
)

for column in all_data.columns:
    if all_data[column].dtype == "object":
        all_data[column] = all_data[column].fillna("None")
    else:
        all_data[column] = all_data[column].fillna(all_data[column].median())

all_data = pd.get_dummies(all_data)

In [5]:
X = all_data.iloc[:len(df_train), :]
X_test = all_data.iloc[len(df_train):, :]

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

In [7]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error

from sklearn.linear_model import Ridge

kf = KFold(n_splits=10, shuffle=True, random_state=42)
ridge = Ridge(alpha=1)

scores = cross_val_score(
    ridge,
    X,
    y,
    scoring="neg_mean_absolute_error",
    cv=kf
)

print("Ridge MAE:", -scores.mean())
ridge.fit(X_scaled, y)
ridge_pred = ridge.predict(X_test_scaled)

Ridge MAE: 0.09129652372098554


In [8]:
from lightgbm import LGBMRegressor

lgb = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

scores = cross_val_score(
    lgb,
    X,
    y,
    scoring="neg_mean_absolute_error",
    cv=kf
)

print("LightGBM MAE:", -scores.mean())

lgb.fit(X, y)
lgb_pred = lgb.predict(X_test)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002708 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3853
[LightGBM] [Info] Number of data points in the train set: 1116, number of used features: 195
[LightGBM] [Info] Start training from score 12.019766
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000974 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3864
[LightGBM] [Info] Number of data points in the train set: 1117, number of used features: 196
[LightGBM] [Info] Start training from score 12.028965
[LightGBM] [Warning] Found

In [9]:
preds = 0.4 * ridge_pred + 0.6 * lgb_pred
final_preds = np.expm1(preds)

sample_submission_df = pd.DataFrame({
    "Id": df_test["Id"],
    "SalePrice": final_preds
})

sample_submission_df.to_csv('/kaggle/working/submission.csv', index=False)